In [1]:
import re
import sys
from pathlib import Path
 
import numpy as np
import pandas as pd
import matplotlib
 
matplotlib.use("Agg")
import matplotlib.pyplot as plt

In [2]:
ROOT = Path(r"C:\Users\limaj\OneDrive\Documentos\UPIIT\INE\INE_Datos_Hackathon\INE_Datos_Hackathon")
OUT_DIR = ROOT.parent / "salida_eda"
 
COLS = ["AELEC", "FELECCION", "EDOCVE", "EDONOM", "MPIOCVE", "MPIONOM", "SECCION",
        "TIPOSEC", "DEF", "DEL", "SEXO", "EDAD", "LN", "SV", "NV", "NS"]
ENCODINGS = ["utf-8", "utf-8-sig", "latin-1"]
 
ESTADOS = {1: "Aguascalientes", 2: "Baja California", 3: "Baja California Sur", 4: "Campeche",
           5: "Coahuila", 6: "Colima", 7: "Chiapas", 8: "Chihuahua", 9: "Ciudad de México",
           10: "Durango", 11: "Guanajuato", 12: "Guerrero", 13: "Hidalgo", 14: "Jalisco",
           15: "Estado de México", 16: "Michoacán", 17: "Morelos", 18: "Nayarit",
           19: "Nuevo León", 20: "Oaxaca", 21: "Puebla", 22: "Querétaro", 23: "Quintana Roo",
           24: "San Luis Potosí", 25: "Sinaloa", 26: "Sonora", 27: "Tabasco", 28: "Tamaulipas",
           29: "Tlaxcala", 30: "Veracruz", 31: "Yucatán", 32: "Zacatecas"}

In [3]:
# Rangos de edad para grupos
BINS = [17, 19, 24, 29, 34, 39, 44, 49, 54, 59, 64, 69, 74, 79, 200]
LABELS = ["18-19", "20-24", "25-29", "30-34", "35-39", "40-44", "45-49", "50-54",
          "55-59", "60-64", "65-69", "70-74", "75-79", "80+"]

In [4]:
# 1. CARGA Y CONSOLIDACIÓN

def leer_csv(path: Path) -> pd.DataFrame:
    """Lee un CSV probando varias codificaciones; DEF/DEL se leen como texto."""
    ultimo = None
    for enc in ENCODINGS:
        try:
            df = pd.read_csv(path, encoding=enc, dtype={"DEF": str, "DEL": str, "FELECCION": str})
            df.columns = [c.strip().upper() for c in df.columns]
            return df
        except UnicodeDecodeError as e:
            ultimo = e
    raise ultimo

def consolidar(root: Path, log: list) -> pd.DataFrame:
    archivos = sorted(p for p in root.rglob("*.csv") if "conteoscensales" in p.name.lower())
    if not archivos:
        sys.exit(f"No se encontraron CSV de conteos censales en {root}")
    log.append(f"Archivos encontrados: {len(archivos)}")
 
    partes = []
    for i, p in enumerate(archivos, 1):
        try:
            df = leer_csv(p)
        except Exception as e:
            log.append(f"[ERROR] {p.name}: {e}")
            continue
 
        faltan = [c for c in COLS if c not in df.columns]
        if faltan:
            log.append(f"[COLUMNAS] {p.name}: faltan {faltan}; extra {[c for c in df.columns if c not in COLS]}")
            continue
        df = df[COLS].copy()
 
        # Año de respaldo desde el nombre del archivo
        m = re.search(r"(20\d\d)", p.name)
        anio_archivo = int(m.group(1)) if m else None

        for c in ["DEF", "DEL"]:
            df[c] = pd.to_numeric(df[c].str.strip().replace("", np.nan), errors="coerce").astype("Int16")
        for c in ["AELEC", "EDOCVE", "MPIOCVE", "SECCION", "SEXO", "EDAD", "LN", "SV", "NV", "NS"]:
            df[c] = pd.to_numeric(df[c], errors="coerce")
        df["TIPOSEC"] = df["TIPOSEC"].astype(str).str.strip().str.upper().str[0]
        df["FELECCION"] = pd.to_datetime(df["FELECCION"], dayfirst=True, errors="coerce")
        df["archivo"] = p.name
    
        # Validaciones por archivo
        n = len(df)
        nulos = int(df[["AELEC", "EDOCVE", "SECCION", "SEXO", "EDAD", "LN", "SV", "NV", "NS"]].isna().any(axis=1).sum())
        ident = int((df.LN != df.SV + df.NV + df.NS).sum())
        anios = df.AELEC.dropna().unique().tolist()
        edos = df.EDOCVE.dropna().unique().tolist()
        flags = []
        if nulos: flags.append(f"nulos={nulos}")
        if ident: flags.append(f"LN!=SV+NV+NS en {ident} filas")
        if anio_archivo and anios != [anio_archivo]: flags.append(f"AELEC {anios} != nombre {anio_archivo}")
        if len(edos) != 1: flags.append(f"entidades={edos}")
        if df.duplicated(["AELEC", "SECCION", "SEXO", "EDAD"]).any(): flags.append("duplicados")
        log.append(f"[{i:03d}/{len(archivos)}] {p.name}: filas={n:,} años={anios} edo={edos} "
                   f"{'OK' if not flags else 'REVISAR -> ' + '; '.join(flags)}")
 
        # Tipos compactos para ahorrar memoria
        for c in ["AELEC", "EDOCVE", "MPIOCVE", "SECCION", "SEXO", "EDAD"]:
            df[c] = df[c].astype("Int32")
        for c in ["LN", "SV", "NV", "NS"]:
            df[c] = df[c].astype("int32")
        for c in ["EDONOM", "MPIONOM", "TIPOSEC", "archivo"]:
            df[c] = df[c].astype("category")
        partes.append(df)
 
    datos = pd.concat(partes, ignore_index=True)
    # Nombre homogéneo de entidad (Ciudad de México cambió de nombre en 2015)
    datos["ESTADO"] = datos["EDOCVE"].map(ESTADOS)
    return datos

In [5]:
# 2. MÉTRICAS

def agrega(df, claves):
    g = df.groupby(claves, observed=True)[["LN", "SV", "NV", "NS"]].sum().reset_index()
    g["PART_BRUTA"] = g.SV / g.LN
    g["PCT_NS"] = g.NS / g.LN
    g["PART"] = g.SV / (g.LN - g.NS).replace(0, np.nan)   # participación ajustada
    return g
 
 
def guardar(df, nombre, tablas):
    df.to_csv(tablas / f"{nombre}.csv", index=False, encoding="utf-8-sig")
 
 
def fig_guardar(nombre, graf):
    plt.tight_layout()
    plt.savefig(graf / f"{nombre}.png", dpi=140)
    plt.close()

In [6]:
# 3. EDA

def eda(d: pd.DataFrame, out: Path):
    tablas, graf = out / "tablas", out / "graficas"
    tablas.mkdir(parents=True, exist_ok=True)
    graf.mkdir(parents=True, exist_ok=True)
    plt.rcParams.update({"axes.grid": True, "grid.alpha": .3, "figure.figsize": (9, 5)})
 
    d["GRUPO_EDAD"] = pd.cut(d.EDAD.astype(float), BINS, labels=LABELS)
    d["SEXO_TXT"] = d.SEXO.map({0: "Hombres", 1: "Mujeres"})
    anios = sorted(d.AELEC.dropna().unique())
 
    # --- Resumen general por año
    anual = agrega(d, ["AELEC"])
    guardar(anual, "01_resumen_por_anio", tablas)
    print("\nResumen nacional por año:\n", anual.round(4).to_string(index=False))
 
    fig, ax = plt.subplots()
    ax.plot(anual.AELEC, anual.PART * 100, "o-", label="Participación ajustada (SV/(LN-NS))")
    ax.plot(anual.AELEC, anual.PART_BRUTA * 100, "s--", label="Participación bruta (SV/LN)")
    ax.set(title="Participación nacional (cuadernillos de casillas aprobadas)", ylabel="%", xticks=anios)
    ax.legend()
    fig_guardar("01_participacion_nacional", graf)
 
    # --- Calidad: % de NS por año y estado
    ns = agrega(d, ["AELEC", "ESTADO"]).pivot(index="ESTADO", columns="AELEC", values="PCT_NS")
    guardar(ns.reset_index(), "02_pct_ns_estado_anio", tablas)
 
    # --- Participación por estado y año (tabla + heatmap)
    pe = agrega(d, ["AELEC", "ESTADO"])
    guardar(pe, "03_participacion_estado_anio", tablas)
    piv = pe.pivot(index="ESTADO", columns="AELEC", values="PART").sort_values(anios[-1])
    fig, ax = plt.subplots(figsize=(8, 10))
    im = ax.imshow(piv.values * 100, aspect="auto", cmap="viridis")
    ax.set(xticks=range(len(piv.columns)), xticklabels=piv.columns, yticks=range(len(piv)),
           yticklabels=piv.index, title="Participación ajustada por estado (%)")
    ax.grid(False)
    for (i, j), v in np.ndenumerate(piv.values * 100):
        if not np.isnan(v):
            ax.text(j, i, f"{v:.0f}", ha="center", va="center", color="w", fontsize=7)
    plt.colorbar(im)
    fig_guardar("03_heatmap_estado_anio", graf)
 
    # --- Por sexo
    ps = agrega(d, ["AELEC", "SEXO_TXT"])
    guardar(ps, "04_participacion_sexo_anio", tablas)
    fig, ax = plt.subplots()
    for s, g in ps.groupby("SEXO_TXT"):
        ax.plot(g.AELEC, g.PART * 100, "o-", label=s)
    ax.set(title="Participación por sexo", ylabel="%", xticks=anios)
    ax.legend()
    fig_guardar("04_participacion_sexo", graf)
 
    # --- Por grupo de edad (curva)
    pg = agrega(d, ["AELEC", "GRUPO_EDAD"])
    guardar(pg, "05_participacion_edad_anio", tablas)
    fig, ax = plt.subplots()
    for a, g in pg.groupby("AELEC"):
        ax.plot(g.GRUPO_EDAD.astype(str), g.PART * 100, "o-", label=str(a))
    ax.set(title="Participación por grupo de edad", ylabel="%")
    ax.tick_params(axis="x", rotation=45)
    ax.legend(title="Elección")
    fig_guardar("05_participacion_edad", graf)
 
    # --- Pirámide de lista nominal (último año)
    u = d[d.AELEC == anios[-1]]
    pir = u.groupby(["GRUPO_EDAD", "SEXO_TXT"], observed=True).LN.sum().unstack()
    fig, ax = plt.subplots()
    ax.barh(pir.index.astype(str), -pir["Hombres"] / 1e6, label="Hombres")
    ax.barh(pir.index.astype(str), pir["Mujeres"] / 1e6, label="Mujeres")
    ax.set(title=f"Pirámide de lista nominal {anios[-1]} (millones)")
    ax.legend()
    fig_guardar("06_piramide_ln", graf)
 
    # --- Tipo de sección
    pt = agrega(d, ["AELEC", "TIPOSEC"])
    guardar(pt, "07_participacion_tiposec_anio", tablas)
    fig, ax = plt.subplots()
    for t, g in pt.groupby("TIPOSEC", observed=True):
        ax.plot(g.AELEC, g.PART * 100, "o-", label={"U": "Urbana", "M": "Mixta", "R": "Rural"}.get(t, t))
    ax.set(title="Participación por tipo de sección", ylabel="%", xticks=anios)
    ax.legend()
    fig_guardar("07_participacion_tiposec", graf)
 
    # --- Municipios y secciones (tablas para consulta)
    guardar(agrega(d, ["AELEC", "EDOCVE", "ESTADO", "MPIOCVE", "MPIONOM"]), "08_participacion_municipio", tablas)
    sec = agrega(d, ["AELEC", "EDOCVE", "ESTADO", "SECCION", "TIPOSEC"])
    guardar(sec, "09_participacion_seccion", tablas)
 
    # --- Distribución de participación por sección (último año)
    s = sec[(sec.AELEC == anios[-1]) & sec.PART.notna()]
    fig, ax = plt.subplots()
    ax.hist(s.PART * 100, bins=60)
    ax.set(title=f"Distribución de participación por sección {anios[-1]}", xlabel="%", ylabel="Secciones")
    fig_guardar("08_hist_secciones", graf)
 
    # --- Alertas de calidad
    top_ns = sec[sec.PCT_NS > .5].groupby(["AELEC", "ESTADO"]).size().rename("secciones_con_NS_mayor_50pct")
    guardar(top_ns.reset_index(), "10_secciones_sin_informacion", tablas)
    edad_ext = d[d.EDAD > 100].groupby("AELEC").LN.sum().rename("LN_edad_mayor_100")
    guardar(edad_ext.reset_index(), "11_edades_extremas", tablas)

In [7]:
def main():
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    log = []
    d = consolidar(ROOT, log)
    log.append(f"\nTotal filas: {len(d):,} | años: {[int(a) for a in sorted(d.AELEC.dropna().unique())]} | "
               f"estados: {d.EDOCVE.nunique()}")
    tabla = d.groupby(["AELEC"], observed=True).EDOCVE.nunique()
    log.append("Estados por año:\n" + tabla.to_string())
    (OUT_DIR / "log_validaciones.txt").write_text("\n".join(log), encoding="utf-8")
    print("\n".join(log[-8:]))
 
    try:
        d.to_parquet(OUT_DIR / "conteos_consolidado.parquet", index=False)
    except ImportError:
        d.to_csv(OUT_DIR / "conteos_consolidado.csv.gz", index=False, compression="gzip")
 
    eda(d, OUT_DIR)
    print(f"\nListo. Resultados en: {OUT_DIR}")

In [8]:
if __name__ == "__main__":
    main()

[187/192] datosabiertos_deceyec_conteoscensales2024_tab.csv: filas=166,627 años=[2024] edo=[27] OK
[188/192] datosabiertos_deceyec_conteoscensales2024_tam.csv: filas=292,663 años=[2024] edo=[28] OK
[189/192] datosabiertos_deceyec_conteoscensales2024_tlax.csv: filas=87,955 años=[2024] edo=[29] OK
[190/192] datosabiertos_deceyec_conteoscensales2024_ver.csv: filas=699,401 años=[2024] edo=[30] OK
[191/192] datosabiertos_deceyec_conteoscensales2024_yuc.csv: filas=163,017 años=[2024] edo=[31] OK
[192/192] datosabiertos_deceyec_conteoscensales2024_zac.csv: filas=229,374 años=[2024] edo=[32] OK

Total filas: 54,872,712 | años: [2009, 2012, 2015, 2018, 2021, 2024] | estados: 32
Estados por año:
AELEC
2009    32
2012    32
2015    32
2018    32
2021    32
2024    32

Resumen nacional por año:
  AELEC       LN       SV       NV      NS  PART_BRUTA  PCT_NS   PART
  2009 77481831 33316974 42298963 1865894      0.4300  0.0241 0.4406
  2012 76490962 47383154 28939159  168649      0.6195  0.0022 0.620